In [ ]:
import pandas as pd
import json
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import ollama
from sentence_transformers import SentenceTransformer

# ========== Embedding model ==========
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ========== Load data ==========
jd_df = pd.read_excel("jobdescription.xlsx")
jd_df["job_id"] = jd_df.index.astype(int)

resume_df = pd.read_csv("Resume.csv")
resume_df = resume_df[["ID", "Resume_str", "Category"]]

# =====================================================================
# Utility: robust JSON loader
# =====================================================================
def safe_json_loads(text):
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            data = [data]
        if isinstance(data, list):
            return data
        return []
    except:
        try:
            start = text.index("[")
            end = text.rindex("]") + 1
            return json.loads(text[start:end])
        except:
            return []

# =====================================================================
# Text summarizer
# =====================================================================
def summarize_text(text, max_length=900):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if len(text) <= max_length:
        return text
    return text[:max_length]

# =====================================================================
# Build Resume text (NEW for your dataset)
# =====================================================================
def build_resume_profile(row):
    return f"""
Resume Text:
{summarize_text(row['Resume_str'])}

Category:
{row['Category']}
"""

# =====================================================================
# Stage 1: Embedding coarse filter
# =====================================================================
def embed_text(text: str):
    return embedder.encode(text if isinstance(text, str) else "")

print("Building JD embeddings...")
jd_texts = jd_df["job_description"].fillna("").tolist()
jd_embeddings = embedder.encode(jd_texts, show_progress_bar=True)

def fast_filter_candidates(resume_text, top_k=20):
    emb = embed_text(resume_text).reshape(1, -1)
    sims = cosine_similarity(emb, jd_embeddings)[0]
    top_ids = np.argsort(sims)[::-1][:top_k]
    return jd_df.iloc[top_ids]

# =====================================================================
# LLM scoring
# =====================================================================
LLM_MODEL = "phi3:mini"

def build_prompt(resume_text, jd_batch):
    jd_section = ""
    for _, row in jd_batch.iterrows():
        desc = summarize_text(row["job_description"], max_length=900)
        jd_section += f"""
---
Job ID: {row['job_id']}
Job Title: {row['job_title']}
Location: {row['location_cleaned']}
Description:
{desc}
"""
    return f"""
You are a senior hiring expert.

Score each job from 0–1 based ONLY on:
1. Skills match
2. Work experience match
3. Education match
4. Location

Output STRICT JSON ONLY:
[
  {{"job_id": 0, "score": 0.00}}
]

Resume:
{resume_text}

Job Descriptions:
{jd_section}

Return TOP 5.
"""

def score_batch_with_llm(resume_text, jd_batch, retry=3):
    prompt = build_prompt(resume_text, jd_batch)
    for _ in range(retry):
        try:
            resp = ollama.chat(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={"num_ctx": 4096}
            )
            txt = resp["message"]["content"]
            data = safe_json_loads(txt)
            if data:
                return data
        except:
            time.sleep(1)
    return []

# =====================================================================
# GUARANTEE top1 exists
# =====================================================================
def ensure_top1(scored, jd_df):
    if len(scored) >= 1:
        return scored[:1]
    fallback_id = int(jd_df.iloc[0]["job_id"])
    return [(fallback_id, 0.0)]

# =====================================================================
# Match per resume
# =====================================================================
def chunk_df(df, size=10):
    for i in range(0, len(df), size):
        yield df.iloc[i:i+size]

def match_resume_to_jobs(resume_text, jd_df):
    filtered = fast_filter_candidates(resume_text, top_k=20)
    scored = {}

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [
            executor.submit(score_batch_with_llm, resume_text, batch)
            for batch in chunk_df(filtered, size=10)
        ]
        for fut in as_completed(futures):
            results = fut.result()
            for r in results:
                try:
                    jid = int(r["job_id"])
                    score = float(r["score"])
                    scored[jid] = max(scored.get(jid, 0.0), score)
                except:
                    continue

    if not scored:
        return ensure_top1([], jd_df)

    ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)
    return ensure_top1(ranked, jd_df)

# =====================================================================
# Run all resumes
# =====================================================================
output = []

for _, row in tqdm(resume_df.iterrows(), total=len(resume_df), desc="Resumes"):
    rid = row["ID"]
    resume_text = build_resume_profile(row)

    top1 = match_resume_to_jobs(resume_text, jd_df)

    jid, score = top1[0]
    job_row = jd_df[jd_df["job_id"] == jid].iloc[0]

    rec = {
        "resume_id": rid,
        "top1_job_id": jid,
        "top1_job_title": job_row["job_title"],
        "top1_job_location": job_row["location_cleaned"],
        "top1_score": score
    }
    output.append(rec)

pd.DataFrame(output).to_excel("resume_job_matching_results.xlsx", index=False)
print("\nDONE! Saved to resume_job_matching_results.xlsx")


Building JD embeddings...


Resumes:   0%|          | 0/2484 [00:00<?, ?it/s]

In [ ]:
# import pandas as pd
# import json
# import numpy as np
# from tqdm import tqdm
# from sklearn.metrics.pairwise import cosine_similarity
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import time
# import ollama
# from sentence_transformers import SentenceTransformer

# # ========== Embedding model ==========
# embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # ========== Load data ==========
# jd_df = pd.read_excel("jobdescription.xlsx")
# jd_df["job_id"] = jd_df.index.astype(int)

# resume_df = pd.read_csv("Resume.csv")
# resume_df = resume_df[
#     [ "ID",
#     "Resume_str", "Category"]
# ]

# # =====================================================================
# # Utility: robust JSON loader
# # =====================================================================
# def safe_json_loads(text):
#     try:
#         data = json.loads(text)
#         if isinstance(data, dict):
#             data = [data]
#         if isinstance(data, list):
#             return data
#         return []
#     except:
#         try:
#             start = text.index("[")
#             end = text.rindex("]") + 1
#             return json.loads(text[start:end])
#         except:
#             return []

# # =====================================================================
# # Text summarizer
# # =====================================================================
# def summarize_text(text, max_length=900):
#     if not isinstance(text, str):
#         return ""
#     text = text.strip()
#     if len(text) <= max_length:
#         return text

#     lines = text.split("\n")
#     summary = []
#     for ln in lines:
#         ln = ln.strip()
#         if len(ln) > 20:
#             summary.append(ln)
#         if len(" ".join(summary)) > max_length:
#             break

#     return " ".join(summary)[:max_length]

# # =====================================================================
# # Build Resume text
# # =====================================================================
# def build_resume_profile(row):
#     return f"""
# Resume Title: {row['Resume Title']}

# Work Experience:
# {summarize_text(row['Work Experience'])}

# Education:
# {summarize_text(row['Education'])}

# Skills:
# {summarize_text(row['Skills'])}

# Additional Information:
# {summarize_text(row['Additional Information'])}
# """

# # =====================================================================
# # Stage 1: Embedding coarse filter
# # =====================================================================
# def embed_text(text: str):
#     return embedder.encode(text if isinstance(text, str) else "")

# print("Building JD embeddings...")
# jd_texts = jd_df["job_description"].fillna("").tolist()
# jd_embeddings = embedder.encode(jd_texts, show_progress_bar=True)

# def fast_filter_candidates(resume_text, top_k=20):
#     emb = embed_text(resume_text).reshape(1, -1)
#     sims = cosine_similarity(emb, jd_embeddings)[0]
#     top_ids = np.argsort(sims)[::-1][:top_k]
#     return jd_df.iloc[top_ids]

# # =====================================================================
# # LLM scoring
# # =====================================================================
# LLM_MODEL = "phi3:mini"

# def build_prompt(resume_text, jd_batch):
#     jd_section = ""
#     for _, row in jd_batch.iterrows():
#         desc = summarize_text(row["job_description"], max_length=900)
#         jd_section += f"""
# ---
# Job ID: {row['job_id']}
# Job Title: {row['job_title']}
# Location: {row['location_cleaned']}
# Description:
# {desc}
# """

#     return f"""
# You are a senior hiring expert.

# Score each job from 0–1 based ONLY on:
# 1. Skills match
# 2. Work experience match
# 3. Education match
# 4. Location

# Output STRICT JSON ONLY:
# [
#   {{"job_id": 0, "score": 0.00}}
# ]

# NO explanations.

# Resume:
# {resume_text}

# Job Descriptions:
# {jd_section}

# Return TOP 5.
# """

# def score_batch_with_llm(resume_text, jd_batch, retry=3):
#     prompt = build_prompt(resume_text, jd_batch)

#     for _ in range(retry):
#         try:
#             resp = ollama.chat(
#                 model=LLM_MODEL,
#                 messages=[{"role": "user", "content": prompt}],
#                 options={"num_ctx": 4096}
#             )
#             txt = resp["message"]["content"]
#             data = safe_json_loads(txt)
#             if data:
#                 return data
#         except:
#             time.sleep(1)

#     return []

# # =====================================================================
# # GUARANTEE top1 exists
# # =====================================================================
# def ensure_top1(scored, jd_df):
#     """
#     Only keep top1. If empty, fill with dummy job.
#     """
#     if len(scored) >= 1:
#         return scored[:1]

#     # pick any job as filler
#     fallback_id = int(jd_df.iloc[0]["job_id"])
#     return [(fallback_id, 0.0)]

# # =====================================================================
# # Match per resume
# # =====================================================================
# def chunk_df(df, size=10):
#     for i in range(0, len(df), size):
#         yield df.iloc[i:i+size]

# def match_resume_to_jobs(resume_text, jd_df):
#     filtered = fast_filter_candidates(resume_text, top_k=20)
#     scored = {}

#     with ThreadPoolExecutor(max_workers=8) as executor:
#         futures = [
#             executor.submit(score_batch_with_llm, resume_text, batch)
#             for batch in chunk_df(filtered, size=10)
#         ]
#         for fut in as_completed(futures):
#             results = fut.result()
#             for r in results:
#                 try:
#                     jid = int(r["job_id"])
#                     score = float(r["score"])
#                     scored[jid] = max(scored.get(jid, 0.0), score)
#                 except:
#                     continue

#     if not scored:
#         return ensure_top1([], jd_df)

#     ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)
#     return ensure_top1(ranked, jd_df)

# # =====================================================================
# # Run all resumes
# # =====================================================================
# output = []

# for _, row in tqdm(resume_df.iterrows(), total=len(resume_df), desc="Resumes"):
#     rid = row["Uniq Id"]
#     resume_text = build_resume_profile(row)

#     top1 = match_resume_to_jobs(resume_text, jd_df)

#     jid, score = top1[0]
#     job_row = jd_df[jd_df["job_id"] == jid].iloc[0]

#     rec = {
#         "resume_id": rid,
#         "top1_job_id": jid,
#         "top1_job_title": job_row["job_title"],
#         "top1_job_location": job_row["location_cleaned"],
#         "top1_score": score
#     }

#     output.append(rec)

# pd.DataFrame(output).to_excel("resume_job_matching_results.xlsx", index=False)
# print("\nDONE! Saved to resume_job_matching_results.xlsx")


'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 114ad6d4-4e87-434b-8eba-2446435aec9c)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Building JD embeddings...


Resumes: 100%|██████████| 10/10 [07:57<00:00, 47.73s/it]


DONE! Saved to resume_job_matching_results.xlsx


In [ ]:
# import pandas as pd
# import json
# import numpy as np
# from tqdm import tqdm
# from sklearn.metrics.pairwise import cosine_similarity
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import time
# import ollama
# from sentence_transformers import SentenceTransformer

# # ========== Embedding model ==========
# embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # ========== Load data ==========
# jd_df = pd.read_excel("jobdescription.xlsx")
# jd_df["job_id"] = jd_df.index.astype(int)

# resume_df = pd.read_csv("resumes.csv")
# resume_df = resume_df[
#     ["Uniq Id", "Resume Title", "Introduction",
#      "Work Experience", "Education",
#      "Skills", "Additional Information"]
# ]

# # =====================================================================
# # Utility: robust JSON loader
# # =====================================================================
# def safe_json_loads(text):
#     """Never crash; always return [] or valid list."""
#     try:
#         data = json.loads(text)
#         if isinstance(data, dict):
#             data = [data]
#         if isinstance(data, list):
#             return data
#         return []
#     except:
#         # Try extract JSON part
#         try:
#             start = text.index("[")
#             end = text.rindex("]") + 1
#             return json.loads(text[start:end])
#         except:
#             return []

# # =====================================================================
# # Text summarizer to reduce prompt size
# # =====================================================================
# def summarize_text(text, max_length=900):
#     """Shorten long resume or job description."""
#     if not isinstance(text, str):
#         return ""

#     text = text.strip()
#     if len(text) <= max_length:
#         return text

#     # Simple extraction (fast)
#     lines = text.split("\n")
#     summary = []
#     for ln in lines:
#         ln = ln.strip()
#         if len(ln) > 20:
#             summary.append(ln)
#         if len(" ".join(summary)) > max_length:
#             break

#     return " ".join(summary)[:max_length]

# # =====================================================================
# # Build Resume text with summarization
# # =====================================================================
# def build_resume_profile(row):
#     return f"""
# Resume Title: {row['Resume Title']}

# Work Experience:
# {summarize_text(row['Work Experience'])}

# Education:
# {summarize_text(row['Education'])}

# Skills:
# {summarize_text(row['Skills'])}

# Additional Information:
# {summarize_text(row['Additional Information'])}
# """

# # =====================================================================
# # Stage 1: Embedding coarse filter
# # =====================================================================
# def embed_text(text: str) -> np.ndarray:
#     return embedder.encode(text if isinstance(text, str) else "")

# print("Building JD embeddings...")
# jd_texts = jd_df["job_description"].fillna("").tolist()
# jd_embeddings = embedder.encode(jd_texts, show_progress_bar=True)

# def fast_filter_candidates(resume_text, top_k=20):
#     emb = embed_text(resume_text).reshape(1, -1)
#     sims = cosine_similarity(emb, jd_embeddings)[0]
#     top_ids = np.argsort(sims)[::-1][:top_k]
#     return jd_df.iloc[top_ids]

# # =====================================================================
# # Stable LLM scoring (with retry + summarization)
# # =====================================================================
# LLM_MODEL = "phi3:mini"

# def build_prompt(resume_text, jd_batch):
#     jd_section = ""
#     for _, row in jd_batch.iterrows():
#         # compress JD text
#         desc = summarize_text(row["job_description"], max_length=900)
#         jd_section += f"""
# ---
# Job ID: {row['job_id']}
# Job Title: {row['job_title']}
# Location: {row['location_cleaned']}
# Description:
# {desc}
# """

#     return f"""
# You are a senior hiring expert.

# Score each job from 0–1 based ONLY on:
# 1. Skills match (highest weight)
# 2. Work experience match
# 3. Education match
# 4. Location match (lowest weight)

# Output STRICT JSON ONLY in this exact format:
# [
#   {{"job_id": 0, "score": 0.00}}
# ]

# NO explanations. NO extra text.

# Resume:
# {resume_text}

# Job Descriptions:
# {jd_section}

# Return TOP 5 highest scoring entries.
# """

# def score_batch_with_llm(resume_text, jd_batch, retry=3):
#     prompt = build_prompt(resume_text, jd_batch)

#     for _ in range(retry):
#         try:
#             resp = ollama.chat(
#                 model=LLM_MODEL,
#                 messages=[{"role": "user", "content": prompt}],
#                 options={"num_ctx": 4096}
#             )
#             txt = resp["message"]["content"]
#             data = safe_json_loads(txt)
#             if data:
#                 return data
        
#         except Exception:
#             time.sleep(1)

#     return []  # fallback

# # =====================================================================
# # Guarantee top1/top2/top3 always exist
# # =====================================================================
# def ensure_top3(scored, jd_df):
#     if len(scored) >= 3:
#         return scored[:3]

#     # fill missing with dummy lowest score
#     needed = 3 - len(scored)

#     all_job_ids = set(jd_df["job_id"])
#     used = {jid for jid, _ in scored}
#     remaining = list(all_job_ids - used)[:needed]

#     for jid in remaining:
#         scored.append((jid, 0.0))

#     return scored[:3]

# # =====================================================================
# # Master function for each resume
# # =====================================================================
# def chunk_df(df, size=10):
#     for i in range(0, len(df), size):
#         yield df.iloc[i:i+size]

# def match_resume_to_jobs(resume_text, jd_df):
#     filtered = fast_filter_candidates(resume_text, top_k=20)
#     scored = {}

#     with ThreadPoolExecutor(max_workers=8) as executor:
#         futures = [
#             executor.submit(score_batch_with_llm, resume_text, batch)
#             for batch in chunk_df(filtered, size=10)
#         ]
#         for fut in as_completed(futures):
#             results = fut.result()
#             for r in results:
#                 try:
#                     jid = int(r["job_id"])
#                     score = float(r["score"])
#                     scored[jid] = max(scored.get(jid, 0.0), score)
#                 except:
#                     continue

#     if not scored:
#         return ensure_top3([], jd_df)

#     ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)
#     return ensure_top3(ranked, jd_df)

# # =====================================================================
# # Run all
# # =====================================================================
# output = []

# for _, row in tqdm(resume_df.iterrows(), total=len(resume_df), desc="Resumes"):
#     rid = row["Uniq Id"]
#     resume_text = build_resume_profile(row)
#     top3 = match_resume_to_jobs(resume_text, jd_df)

#     rec = {"resume_id": rid}
#     for i, (jid, score) in enumerate(top3, start=1):
#         job_row = jd_df[jd_df["job_id"] == jid].iloc[0]
#         rec[f"top{i}_job_id"] = jid
#         rec[f"top{i}_job_title"] = job_row["job_title"]
#         rec[f"top{i}_job_location"] = job_row["location_cleaned"]
#         rec[f"top{i}_score"] = score

#     output.append(rec)

# pd.DataFrame(output).to_excel("resume_job_matching_results.xlsx", index=False)
# print("\nDONE! Saved to resume_job_matching_results.xlsx")


Building JD embeddings...


Resumes: 100%|██████████| 10/10 [07:50<00:00, 47.03s/it]


DONE! Saved to resume_job_matching_results.xlsx
